# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [2]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU (UUID: GPU-a2f32238-420d-3493-bafe-90f1e473bebc)


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [5]:
!python scripts/prepare_data.py --kaggle

Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT

dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}



  0%|          | 0.00/185M [00:00<?, ?B/s]
  1%|          | 1.00M/185M [00:00<01:41, 1.90MB/s]
  1%|          | 2.00M/185M [00:00<00:58, 3.27MB/s]
  2%|▏         | 4.00M/185M [00:00<00:33, 5.61MB/s]
  3%|▎         | 5.00M/185M [00:01<00:29, 6.32MB/s]
  4%|▍         | 7.00M/185M [00:01<00:22, 8.23MB/s]
  5%|▍         | 9.00M/185M [00:01<00:20, 9.02MB/s]
  5%|▌         | 10.0M/185M [00:01<00:20, 8.83MB/s]
  6%|▌         | 11.0M/185M [00:01<00:21, 8.54MB/s]
  7%|▋         | 13.0M/185M [00:01<00:17, 10.5MB/s]
  8%|▊         | 15.0M/185M [00:02<00:17, 10.0MB/s]
  9%|▊         | 16.0M/185M [00:02<00:20, 8.86MB/s]
  9%|▉         | 17.0M/185M [00:02<00:19, 8.89MB/s]
 10%|█         | 19.0M/185M [00:02<00:20, 8.46MB/s]
 11%|█         | 20.0M/185M [00:02<00:21, 8.20MB/s]
 11%|█▏        | 21.0M/185M [00:02<00:20, 8.23MB/s]
 12%|█▏        | 22.0M/185M [00:02<00:20, 8.18MB/s]
 12%|█▏        | 23.0M/185M [00:03<00:22, 7.48MB/s]
 13%|█▎        | 24.0M/185M [00:03<00:23, 7.25MB/s]
 14%|█▍        | 26.

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# Chay duoc khi thu muc hien tai la thu muc lab hoac thu muc cha.
lab_dir = Path.cwd()
if not (lab_dir / "data/splits/train.jsonl").exists():
    lab_dir = lab_dir / "K4-Track-4-Day10-VisionLanguageModel"
data_dir = lab_dir / "data"

splits = {}
for split in ("train", "val", "test"):
    with (data_dir / "splits" / f"{split}.jsonl").open(encoding="utf-8") as f:
        splits[split] = [json.loads(line) for line in f if line.strip()]
    counts = Counter(obj["label"] for row in splits[split] for obj in row["objects"])
    print(f"{split}: {len(splits[split])} images, {sum(counts.values())} boxes | {dict(counts)}")

# Moi phien quay chi duoc xuat hien trong mot tap du lieu.
sessions = {name: {row["session"] for row in rows} for name, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = sessions[a] & sessions[b]
    assert not overlap, f"Overlapping sessions in {a}/{b}: {sorted(overlap)}"
print("OK: train/val/test have no overlapping sessions.")

train_records = splits["train"]
print("\nFirst 3 train records:")
for row in train_records[:3]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

# Chon 6 anh co nhan; seed giup lan chay sau chon lai cung cac anh.
labeled_records = [row for row in train_records if row["objects"]]
if not labeled_records:
    raise ValueError("No labeled images found in train.jsonl")
samples = random.Random(42).sample(labeled_records, min(6, len(labeled_records)))
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax in axes.flat:
    ax.axis("off")

for ax, row in zip(axes.flat, samples):
    with Image.open(data_dir / row["image_path"]) as image:
        image = image.convert("RGB")
    width, height = image.size
    ax.imshow(image)
    for obj in row["objects"]:
        # YOLO: tam (xc, yc), rong w, cao h deu nam tren thang 0-1.
        # Doi sang toa do pixel de ve khung len anh.
        xc, yc, w, h = obj["yolo"]
        x = (xc - w / 2) * width
        y = (yc - h / 2) * height
        box_width, box_height = w * width, h * height
        color = colors.get(obj["label"], "lime")
        ax.add_patch(Rectangle((x, y), box_width, box_height,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x, y, obj["label"], color=color, fontsize=10,
                va="bottom", bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
    ax.set_title(f"{row['image_id']} | {len(row['objects'])} boxes", fontsize=9)

plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json


def yolo_to_box(label, yolo):
    """Convert YOLO center/size (0-1) to corner coordinates (0-1000)."""
    xc, yc, w, h = yolo
    corners = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]
    bbox = [max(0, min(1000, round(value * 1000))) for value in corners]
    return {"label": label, "bbox_2d": bbox}


def objects_to_answer(objects):
    """Return compact JSON, sorted from top to bottom, then left to right."""
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda box: (box["bbox_2d"][1], box["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))


PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Run the data cell above first: it defines train_records, data_dir and plotting imports.
example = next(row for row in train_records if row["objects"])
answer = objects_to_answer(example["objects"])
print("Image:", example["image_id"])
print("Answer:", answer)

# Compare original YOLO boxes with boxes converted from JSON back to pixels.
with Image.open(data_dir / example["image_path"]) as image:
    image = image.convert("RGB")
width, height = image.size
original_boxes = []
for obj in example["objects"]:
    xc, yc, w, h = obj["yolo"]
    original_boxes.append({
        "label": obj["label"],
        "bbox_2d": [(xc - w / 2) * width, (yc - h / 2) * height,
                    (xc + w / 2) * width, (yc + h / 2) * height],
    })
restored_boxes = [
    {"label": box["label"],
     "bbox_2d": [value / 1000 * size for value, size in
                 zip(box["bbox_2d"], (width, height, width, height))]}
    for box in json.loads(answer)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, boxes, title in zip(axes, (original_boxes, restored_boxes),
                            ("Original YOLO", "JSON 0-1000 -> pixels")):
    ax.imshow(image)
    for box in boxes:
        x1, y1, x2, y2 = box["bbox_2d"]
        color = colors.get(box["label"], "lime")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, y1, box["label"], color=color, va="bottom",
                bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. E0: zero-shot

In [ ]:
import json

import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

# First run the cells defining splits, lab_dir, data_dir and PROMPT.
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU: Runtime > Change runtime type > T4 GPU.")

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BATCH_SIZE = 4  # Reduce to 1 or 2 if GPU memory is insufficient.
test_records = splits["test"]
if not test_records:
    raise ValueError("The test split is empty. Run data preparation first.")

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Keep vision/projector and output head in fp16 for later training.
    llm_int8_skip_modules=["visual", "lm_head"],
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

output_path = lab_dir / "work" / "e0.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)
preview = []

# Start a fresh file each run. Save raw answers, without repairing their JSON.
with output_path.open("w", encoding="utf-8") as output_file:
    for start in tqdm(range(0, len(test_records), BATCH_SIZE), desc="E0 inference"):
        batch = test_records[start:start + BATCH_SIZE]
        conversations = []
        for row in batch:
            with Image.open(data_dir / row["image_path"]) as image:
                rgb_image = image.convert("RGB")
            conversations.append([
                {"role": "user", "content": [
                    {"type": "image", "image": rgb_image},
                    {"type": "text", "text": PROMPT},
                ]}
            ])

        inputs = processor.apply_chat_template(
            conversations,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        ).to(model.device)
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
            )
        # Remove the entire padded input, leaving only generated answer tokens.
        answer_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
        raw_answers = processor.batch_decode(
            answer_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        for row, raw in zip(batch, raw_answers):
            prediction = {"image_id": row["image_id"], "raw": raw}
            output_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
            if len(preview) < 5:
                preview.append(prediction)
        output_file.flush()
        del inputs, generated_ids, answer_ids

print(f"Saved {len(test_records)} predictions to {output_path}")
print("First 5 raw outputs:")
for prediction in preview:
    print("\nImage:", prediction["image_id"])
    print(prediction["raw"])


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence


def encode_example(image, prompt, answer):
    """Encode image/chat; supervise only the answer and <|im_end|>."""
    user = {
        "role": "user",
        "content": [
            {"type": "image", "image": image.convert("RGB")},
            {"type": "text", "text": prompt},
        ],
    }
    options = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt_enc = processor.apply_chat_template(
        [user], add_generation_prompt=True, **options,
    )
    full_enc = processor.apply_chat_template(
        [user, {"role": "assistant", "content": answer}],
        add_generation_prompt=False, **options,
    )

    n = prompt_enc["input_ids"].shape[1]
    input_ids = full_enc["input_ids"]
    assert torch.equal(input_ids[:, :n], prompt_enc["input_ids"]), (
        "Prompt is not a prefix of the full conversation; check the chat template."
    )

    labels = input_ids.clone()
    labels[:, :n] = -100  # Ignore prompt, image tokens and assistant header.

    # Ignore any newline after the answer-ending token.
    end_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    end_positions = (input_ids[0, n:] == end_id).nonzero(as_tuple=True)[0]
    assert end_positions.numel() > 0, "Answer is missing <|im_end|>."
    end = n + end_positions[0].item()
    labels[:, end + 1:] = -100
    full_enc["labels"] = labels
    return full_enc


def collate(batch):
    """Pad token sequences and concatenate image data into a batch."""
    if not batch:
        raise ValueError("Batch must not be empty.")
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        raise ValueError("Tokenizer has no pad_token_id.")

    # Right-pad training batches independently of E0 padding_side="left".
    padding_values = {
        "input_ids": pad_id,
        "attention_mask": 0,
        "labels": -100,
    }
    if any("mm_token_type_ids" in example for example in batch):
        if not all("mm_token_type_ids" in example for example in batch):
            raise ValueError("All examples must provide mm_token_type_ids.")
        padding_values["mm_token_type_ids"] = 0

    result = {
        key: pad_sequence(
            [example[key].squeeze(0) for example in batch],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding_values.items()
    }
    for key in ("pixel_values", "image_grid_thw"):
        result[key] = torch.cat([example[key] for example in batch], dim=0)
    return result


# Check one training image using the variables from steps 1-3.
example = next(row for row in train_records if row["objects"])
answer = objects_to_answer(example["objects"])
with Image.open(data_dir / example["image_path"]) as image:
    image = image.convert("RGB")
encoded = encode_example(image, PROMPT, answer)

image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
num_image_tokens = (encoded["input_ids"] == image_token_id).sum().item()
print(f"Image size: {image.size} | Image tokens: {num_image_tokens}")
if image.size == (640, 360):
    assert num_image_tokens == 220, f"Expected 220 image tokens, got {num_image_tokens}."

supervised_ids = encoded["labels"][encoded["labels"] != -100]
decoded_answer = processor.tokenizer.decode(
    supervised_ids.tolist(), skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)
print("Tokens used for loss:", decoded_answer)
assert decoded_answer == answer + "<|im_end|>", (
    "Supervised tokens must decode to the JSON answer plus <|im_end|>."
)

# Check padding and image concatenation with two different answer lengths.
empty_encoded = encode_example(image, PROMPT, "[]")
check_batch = collate([encoded, empty_encoded])
for key in ("attention_mask", "labels", "mm_token_type_ids"):
    if key in check_batch:
        assert check_batch[key].shape == check_batch["input_ids"].shape
assert torch.all(check_batch["labels"][check_batch["attention_mask"] == 0] == -100)
for key in ("pixel_values", "image_grid_thw"):
    assert check_batch[key].shape[0] == encoded[key].shape[0] + empty_encoded[key].shape[0]
print("OK: prompt mask, answer, padding and image concatenation.")
del check_batch, empty_encoded


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re

import torch
from peft import LoraConfig, PeftModel, TaskType, get_peft_model


def apply_setup(model, setup):
    """Freeze base weights and configure trainable weights for E1, E2 or E3."""
    setup = setup.strip().upper()
    if setup not in {"E1", "E2", "E3"}:
        raise ValueError("setup must be E1, E2 or E3.")
    if isinstance(model, PeftModel) or getattr(model, "peft_config", None):
        raise ValueError("Load a fresh base model for each experiment before apply_setup.")

    llm_pattern = (
        r"model\.language_model\.layers\.\d+\."
        r"(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
    )
    vision_pattern = (
        r"model\.visual\.blocks\.\d+\."
        r"(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"
    )
    target_pattern = llm_pattern
    if setup == "E3":
        target_pattern = rf"(?:{llm_pattern}|{vision_pattern})"

    matched_modules = [
        name for name, _ in model.named_modules()
        if re.fullmatch(target_pattern, name)
    ]
    expected = 348 if setup == "E3" else 252
    if len(matched_modules) != expected:
        raise ValueError(
            f"{setup}: matched {len(matched_modules)} LoRA modules, expected {expected}. "
            "Check the model architecture and module names."
        )

    for parameter in model.parameters():
        parameter.requires_grad = False
    # Keep gradients flowing through frozen embeddings during checkpointing.
    model.enable_input_require_grads()

    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_pattern,
        bias="none",
    )
    peft_model = get_peft_model(model, config)

    if setup in {"E2", "E3"}:
        for name, parameter in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                parameter.data = parameter.data.to(dtype=torch.float32)
                parameter.requires_grad = True

    print(f"{setup}: {len(matched_modules)} LoRA target modules")
    peft_model.print_trainable_parameters()
    return peft_model


# Run with a freshly loaded base model for each experiment:
# model = apply_setup(model, "E1")  # LLM LoRA only.
# model = apply_setup(model, "E2")  # LLM LoRA + train all merger weights.
# model = apply_setup(model, "E3")  # LLM/vision LoRA + train all merger weights.


## 6. Vòng huấn luyện

In [ ]:
import gc
import math
import time

import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

# Mot lan chay cell: train ca E1, E2, E3, moi setup tu model goc moi.
EXPERIMENTS = ("E1", "E2", "E3")
MAX_STEPS = 300  # Moi setup co 300 lan cap nhat trong so.
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM = 4  # Batch hieu dung: 8 anh.
LR = 2e-4
SEED = 42
SAVE_EVERY = 100

if not torch.cuda.is_available():
    raise RuntimeError("Training requires a CUDA GPU (Colab T4).")
if not train_records:
    raise ValueError("The training split is empty.")

# Bo model E0/bo toi uu cu; chi giu mot model tren GPU tai moi thoi diem.
for variable in ("scheduler", "optimizer", "params", "scaler", "model"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()
processor = AutoProcessor.from_pretrained(MODEL_ID)


def load_fresh_base(model_id):
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"],
    )
    return Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quantization,
        dtype=torch.float16, device_map={"": 0},
    )


class RoadDamageDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(data_dir / row["image_path"]) as image:
            return encode_example(image, PROMPT, objects_to_answer(row["objects"]))


def train_setup(setup, records=None, run_name=None):
    records = train_records if records is None else records
    if not records:
        raise ValueError("The training subset is empty.")
    run_dir = lab_dir / "work" / (run_name or setup.lower())
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    model = load_fresh_base(MODEL_ID)
    model = apply_setup(model, setup)
    loader = DataLoader(
        RoadDamageDataset(records),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )
    params = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not params:
        raise ValueError("No trainable parameters. Run apply_setup first.")
    if any(parameter.dtype == torch.float16 for parameter in params):
        raise ValueError("GradScaler requires trainable weights in float32; check step 5.")
    trainable_params = sum(parameter.numel() for parameter in params)
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    warmup_steps = max(1, math.ceil(MAX_STEPS * 0.05))


    def lr_multiplier(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, MAX_STEPS - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
    scaler = torch.amp.GradScaler("cuda")
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model.train()
    optimizer.zero_grad(set_to_none=True)
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.cuda.synchronize(model.device)
    torch.cuda.reset_peak_memory_stats(model.device)
    started = time.perf_counter()
    loss_history = []


    def save_checkpoint(destination, completed_steps):
        destination.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(destination)  # Adapter LoRA.
        processor.save_pretrained(destination)
        # PEFT khong luu cac trong so projector duoc huan luyen day du.
        if setup in {"E2", "E3"}:
            extra_trainable = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad and
                ("visual.merger" in name or "visual.deepstack_merger_list" in name)
            }
            if not extra_trainable:
                raise RuntimeError("Missing trainable projector weights for E2/E3.")
            # Nap vao PeftModel bang load_state_dict(..., strict=False) o buoc 7.
            torch.save(extra_trainable, destination / "extra_trainable.pt")
        torch.cuda.synchronize(model.device)
        meta = {
            "setup": setup,
            "model_id": MODEL_ID,
            "steps": completed_steps,
            "n_train_images": len(records),
            "trainable_params": trainable_params,
            "peak_vram_gb": torch.cuda.max_memory_allocated(model.device) / 1024**3,
            "train_minutes": (time.perf_counter() - started) / 60,
            "batch_size": TRAIN_BATCH_SIZE,
            "grad_accum": GRAD_ACCUM,
            "lr": LR,
            "seed": SEED,
            "loss_history": loss_history,
        }
        (destination / "run.json").write_text(
            json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8",
        )
        return meta


    batches = iter(loader)
    step = 0
    skipped_steps = 0
    progress_bar = tqdm(total=MAX_STEPS, desc=f"Train {setup}")
    while step < MAX_STEPS:
        step_loss = 0.0
        for micro_step in range(GRAD_ACCUM):
            try:
                batch = next(batches)
            except StopIteration:
                batches = iter(loader)  # Het du lieu: sang luot moi, tiep tuc tron anh.
                batch = next(batches)
            batch = {key: value.to(model.device) for key, value in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = model(**batch).loss
            if not torch.isfinite(loss).item():
                optimizer.zero_grad(set_to_none=True)
                raise FloatingPointError(
                    f"Non-finite loss at step {step + 1}, micro-batch {micro_step + 1}; "
                    "stop training and check fp16 overflow."
                )
            step_loss += loss.detach().item() / GRAD_ACCUM
            scaler.scale(loss / GRAD_ACCUM).backward()
            del batch, loss

        for name, parameter in model.named_parameters():
            if parameter.grad is not None and parameter.grad.dtype == torch.float16:
                print(
                    "FP16 gradient:", name,
                    "| weight dtype:", parameter.dtype,
                    "| requires_grad:", parameter.requires_grad,
                    flush=True,
                )

        scaler.unscale_(optimizer)
        # GradScaler tu giam scale khi gradient tran; chi dem buoc cap nhat thanh cong.
        previous_scale = scaler.get_scale()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        if scaler.get_scale() < previous_scale:
            skipped_steps += 1
            if skipped_steps >= 20:
                raise FloatingPointError("20 consecutive gradient overflows; check fp16 stability.")
            tqdm.write("Gradient overflow: reduced scale, retrying optimizer step.")
            continue
        skipped_steps = 0
        step += 1
        scheduler.step()
        progress_bar.update(1)
        loss_history.append({"step": step, "loss": step_loss})
        if step == 1 or step % 10 == 0:
            tqdm.write(f"step={step}/{MAX_STEPS} loss={step_loss:.4f} "
                       f"lr={optimizer.param_groups[0]['lr']:.2e}")
        if step % SAVE_EVERY == 0:
            save_checkpoint(run_dir / f"checkpoint-{step}", step)
    progress_bar.close()

    # Ban cuoi nam ngay trong work/eN; checkpoint moi 100 buoc o thu muc con.
    run_meta = save_checkpoint(run_dir, MAX_STEPS)
    model.eval()
    print(f"Saved adapter and run.json to {run_dir}")
    print(json.dumps({key: run_meta[key] for key in
                      ("trainable_params", "peak_vram_gb", "train_minutes")}, indent=2))
    return run_meta


training_runs = {}
for setup in EXPERIMENTS:
    print(f"\n=== Training {setup} ===")
    training_runs[setup] = train_setup(setup)
    # Ham da ket thuc: model, optimizer va gradient cua setup truoc duoc giai phong.
    gc.collect()
    torch.cuda.empty_cache()
print("Finished training E1, E2, E3. Run step 7 to create all six result files.")


## 7. Suy luận và chấm E1–E3

In [ ]:
import gc
import json
import math
import subprocess
import sys

import torch
from PIL import Image
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

# Mot lan chay cell: du doan va cham test cho ca ba setup da train o buoc 6.
EVAL_SETUPS = ("E1", "E2", "E3")
INFER_BATCH_SIZE = 2  # Giam con 1 neu GPU khong du bo nho.

# Ho tro chay lai cell sau mot phien notebook cu.
for variable in ("scheduler", "optimizer", "params", "scaler", "model"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()


def evaluate_setup(setup, run_name=None, score_name=None):
    global processor  # encode_example/collate cung su dung processor nay.
    run_name = run_name or setup.lower()
    adapter_dir = (lab_dir / "work" / run_name).resolve()
    output_path = (lab_dir / "work" / f"{run_name}.jsonl").resolve()
    evidence_path = (lab_dir / "report" / "evidence" / f"{run_name}.json").resolve()

    if setup not in {"E1", "E2", "E3"}:
        raise ValueError("setup must be E1, E2 or E3.")
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU (Colab T4).")
    for filename in ("adapter_config.json", "run.json"):
        if not (adapter_dir / filename).is_file():
            raise FileNotFoundError(f"Missing {adapter_dir / filename}; finish step 6 first.")
    run_meta = json.loads((adapter_dir / "run.json").read_text(encoding="utf-8"))
    if run_meta.get("setup") != setup:
        raise ValueError("The saved run.json does not match setup.")
    meta_keys = ("trainable_params", "peak_vram_gb", "train_minutes")
    for key in meta_keys:
        value = run_meta.get(key)
        if (isinstance(value, bool) or not isinstance(value, (int, float))
                or not math.isfinite(value) or value <= 0):
            raise ValueError(f"Invalid {key} in run.json: {value!r}")
    projector_path = adapter_dir / "extra_trainable.pt"
    if setup in {"E2", "E3"} and not projector_path.is_file():
        raise FileNotFoundError(f"Missing trained projector: {projector_path}")
    test_records = splits["test"]
    if not test_records:
        raise ValueError("The test split is empty.")

    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"],
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        run_meta["model_id"], quantization_config=quantization,
        dtype=torch.float16, device_map={"": 0},
    )
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    if setup in {"E2", "E3"}:
        extra_trainable = torch.load(projector_path, map_location="cpu", weights_only=True)
        projector_params = {
            name: parameter for name, parameter in model.named_parameters()
            if "visual.merger" in name or "visual.deepstack_merger_list" in name
        }
        if not projector_params or set(extra_trainable) != set(projector_params):
            raise ValueError("Saved projector keys do not match the reloaded model.")
        for name, parameter in projector_params.items():
            if extra_trainable[name].shape != parameter.shape:
                raise ValueError(f"Projector shape mismatch: {name}")
            # Giu float32 nhu luc train; autocast xu ly phep tinh fp16.
            parameter.data = parameter.data.to(dtype=torch.float32)
        model.load_state_dict(extra_trainable, strict=False)
        del extra_trainable, projector_params, parameter
    model.eval()
    model.config.use_cache = True
    processor = AutoProcessor.from_pretrained(adapter_dir)
    processor.tokenizer.padding_side = "left"

    output_path.parent.mkdir(parents=True, exist_ok=True)
    preview = []
    # Ghi cau tra loi tho, khong sua JSON truoc khi cham diem.
    with output_path.open("w", encoding="utf-8") as output_file:
        for start in tqdm(range(0, len(test_records), INFER_BATCH_SIZE),
                          desc=f"{setup} test inference"):
            batch = test_records[start:start + INFER_BATCH_SIZE]
            conversations = []
            for row in batch:
                with Image.open(data_dir / row["image_path"]) as image:
                    rgb_image = image.convert("RGB")
                conversations.append([
                    {"role": "user", "content": [
                        {"type": "image", "image": rgb_image},
                        {"type": "text", "text": PROMPT},
                    ]}
                ])
            inputs = processor.apply_chat_template(
                conversations, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True,
            ).to(model.device)
            with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
                generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            answer_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
            raw_answers = processor.batch_decode(
                answer_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False,
            )
            for row, raw in zip(batch, raw_answers):
                prediction = {"image_id": row["image_id"], "raw": raw}
                output_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
                if len(preview) < 5:
                    preview.append(prediction)
            output_file.flush()
            del inputs, generated_ids, answer_ids

    print(f"Saved {len(test_records)} predictions to {output_path}")
    for prediction in preview:
        print("\nImage:", prediction["image_id"])
        print(prediction["raw"])

    # --meta lay so do luc TRAIN tu run.json, khong do lai luc suy luan.
    command = [
        sys.executable, str((lab_dir / "scripts" / "score.py").resolve()),
        "--pred", str(output_path), "--name", score_name or setup, "--out", str(evidence_path),
        "--splits-dir", str((data_dir / "splits").resolve()), "--split", "test",
    ]
    for key in meta_keys:
        value = run_meta[key]
        # score.py nhan so thuc co dau cham; tranh ky hieu khoa hoc nhu 1e-05.
        formatted = str(value) if isinstance(value, int) else f"{value:.16f}"
        command.extend(["--meta", f"{key}={formatted}"])
    result = subprocess.run(command, cwd=str(lab_dir.resolve()),
                            capture_output=True, text=True, encoding="utf-8")
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
    scores = json.loads(evidence_path.read_text(encoding="utf-8"))
    print(f"Saved scores to {evidence_path}")
    return {"predictions": str(output_path), "scores": str(evidence_path)}


evaluation_files = {}
for setup in EVAL_SETUPS:
    print(f"\n=== Evaluating {setup} ===")
    evaluation_files[setup] = evaluate_setup(setup)
    gc.collect()
    torch.cuda.empty_cache()
print("\nSix output files:")
for setup, files in evaluation_files.items():
    print(f"{setup}: {files['predictions']}")
    print(f"{setup}: {files['scores']}")


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# GUIDE bước 8: chỉ đổi data_size, không thực hiện bước 9.
import copy
import gc
import json
import math
import random
import shutil

# Chạy các cell bước 1-7 trước. Dùng F1 đã chấm để chọn setup gốc.
evidence_dir = lab_dir / "report" / "evidence"
baseline_scores = {}
for setup in ("E1", "E2", "E3"):
    path = evidence_dir / f"{setup.lower()}.json"
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; finish step 7 first.")
    score = json.loads(path.read_text(encoding="utf-8"))
    if (score.get("split") != "test" or score.get("coord") != "norm1000"
            or score.get("n_images") != len(splits["test"])
            or score.get("missing_predictions") != 0):
        raise ValueError(f"{setup}: expected complete test scores on norm1000 coordinates.")
    f1 = score["micro"]["f1"]
    if not isinstance(f1, (int, float)) or not math.isfinite(f1):
        raise ValueError(f"{setup}: invalid F1.")
    baseline_scores[setup] = score

# Nếu F1 bằng nhau, ưu tiên E1 rồi E2 rồi E3.
best_setup = max(baseline_scores, key=lambda setup: baseline_scores[setup]["micro"]["f1"])
baseline_dir = lab_dir / "work" / best_setup.lower()
baseline_meta = json.loads((baseline_dir / "run.json").read_text(encoding="utf-8"))
# Bảo đảm các thiết lập hiện tại khớp lần train 100% đã lưu.
expected = {"setup": best_setup, "model_id": MODEL_ID, "steps": MAX_STEPS,
            "batch_size": TRAIN_BATCH_SIZE, "grad_accum": GRAD_ACCUM,
            "lr": LR, "seed": SEED}
for key, value in expected.items():
    if baseline_meta.get(key) != value:
        raise ValueError(f"Restore {key}={baseline_meta.get(key)!r} from the baseline before E4.")
if baseline_meta.get("n_train_images", len(train_records)) != len(train_records):
    raise ValueError("The baseline was not trained on the current full training split.")
baseline_predictions = lab_dir / "work" / f"{best_setup.lower()}.jsonl"
if not baseline_predictions.is_file():
    raise FileNotFoundError(baseline_predictions)

# Lấy ngẫu nhiên 10% số ảnh, gồm cả ảnh không có đối tượng; không lấy từ val/test.
subset_size = max(1, math.ceil(len(train_records) * 0.1))
indices = sorted(random.Random(SEED).sample(range(len(train_records)), subset_size))
subset_records = [train_records[index] for index in indices]
run_name = "e4_data_size_0.1"
subset_dir = lab_dir / "work" / run_name
subset_dir.mkdir(parents=True, exist_ok=True)
(subset_dir / "train_subset.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in subset_records),
    encoding="utf-8",
)
print(f"Best setup: {best_setup}, F1={baseline_scores[best_setup]['micro']['f1']:.4f}")
print(f"Train images: 10%={subset_size}, 100%={len(train_records)}; seed={SEED}")
print(f"Both use {MAX_STEPS} optimizer steps and identical model, LoRA, LR, batch and prompt.")

# 100% chính là setup gốc: dùng lại kết quả, không train/chấm test lần nữa.
full_name = "e4_data_size_1.0"
full_prediction_path = lab_dir / "work" / f"{full_name}.jsonl"
shutil.copyfile(baseline_predictions, full_prediction_path)
full_score = copy.deepcopy(baseline_scores[best_setup])
full_score["name"] = "E4_data_size_1.0"
full_score["meta"].update({"base_setup": best_setup, "ablation_variable": "data_size",
                           "train_fraction": 1.0, "n_train_images": len(train_records),
                           "seed": SEED, "source_run": str(baseline_dir)})
full_evidence_path = evidence_dir / f"{full_name}.json"
full_evidence_path.write_text(json.dumps(full_score, ensure_ascii=False, indent=2), encoding="utf-8")

# Train 10% từ model gốc mới, không tiếp tục adapter của bản 100%.
for variable in ("scheduler", "optimizer", "params", "scaler", "model"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()
small_meta = train_setup(best_setup, records=subset_records, run_name=run_name)
small_meta.update({"base_setup": best_setup, "ablation_variable": "data_size",
                   "train_fraction": 0.1})
(subset_dir / "run.json").write_text(
    json.dumps(small_meta, ensure_ascii=False, indent=2), encoding="utf-8",
)
gc.collect()
torch.cuda.empty_cache()
small_files = evaluate_setup(best_setup, run_name=run_name, score_name="E4_data_size_0.1")
gc.collect()
torch.cuda.empty_cache()
small_evidence_path = Path(small_files["scores"])
small_score = json.loads(small_evidence_path.read_text(encoding="utf-8"))
small_score["meta"].update({"base_setup": best_setup, "ablation_variable": "data_size",
                            "train_fraction": 0.1, "n_train_images": subset_size, "seed": SEED})
small_evidence_path.write_text(
    json.dumps(small_score, ensure_ascii=False, indent=2), encoding="utf-8",
)

e4_results = {"0.1": small_score, "1.0": full_score}
print("\nData size | Images | F1     | CI95          | FP/image | VRAM GB | Minutes")
for fraction, score in e4_results.items():
    lo, hi = score["f1_ci95"]
    meta = score["meta"]
    print(f"{float(fraction):>8.0%} | {meta['n_train_images']:>6} | "
          f"{score['micro']['f1']:.4f} | [{lo:.4f}, {hi:.4f}] | "
          f"{score['fp_per_image']:.4f}   | {meta['peak_vram_gb']:.2f}    | "
          f"{meta['train_minutes']:.2f}")
delta = full_score["micro"]["f1"] - small_score["micro"]["f1"]
print(f"\nF1(100%) - F1(10%) = {delta:+.4f}")
small_lo, small_hi = small_score["f1_ci95"]
full_lo, full_hi = full_score["f1_ci95"]
if max(small_lo, full_lo) <= min(small_hi, full_hi):
    print("CI95 chồng lấn: chưa đủ cơ sở để khẳng định chắc chắn bản nào tốt hơn.")
else:
    print("CI95 không chồng lấn: kết quả gợi ý có khác biệt, nhưng chưa phải kiểm định chênh lệch F1.")
print(f"Test có {full_score['n_images']} ảnh; kết luận chỉ áp dụng cho tập này và seed đã dùng.")
print("Setup gốc được chọn theo F1 test đã có, nên so sánh này mang tính thăm dò.")
print("Evidence:", small_evidence_path, full_evidence_path, sep="\n")
